In [1]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Load env variables and connect to database
load_dotenv("../.env")
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DATABASE_URL)

# Read cleaned data
df = pd.read_sql("SELECT * FROM cleaned_housing_data", engine)


def feature_engineering(df):
    """
    Create new features and improve existing ones
    based on EDA findings.
    """
    
    df = df.copy()

    # House age: older houses may have different prices
    df['house_age'] = df['yr_sold'] - df['year_built']

    # Total living area: combine main floors + basement
    df['total_living_area'] = (
        df['first_flr_sf'] + df['second_flr_sf'] + df['total_bsmt_sf']
    )

    # Check if house was remodeled (1 = yes, 0 = no)
    df['is_remodeled'] = (df['year_built'] != df['year_remod_add']).astype(int)

    # Total bathrooms (half bath = 0.5)
    df['total_bathrooms'] = (
        df['full_bath'] + df['half_bath'] * 0.5 +
        df['bsmt_full_bath'] + df['bsmt_half_bath'] * 0.5
    )

    # Some columns have many zeros
    # create simple flags: does this feature exist or not
    zero_cols = ['mas_vnr_area', 'wood_deck_sf', 'open_porch_sf', 'bsmt_fin_sf_1']
    
    for col in zero_cols:
        if col in df.columns:
            df[f'has_{col}'] = (df[col] > 0).astype(int)

    # Apply log transform to reduce skew (handle large values better)
    # log1p is safe for zero values
    log_cols = [
        'gr_liv_area', 'lot_frontage', 'total_living_area',
        'garage_area', 'bsmt_unf_sf', 'lot_area'
    ] + zero_cols

    for col in log_cols:
        if col in df.columns:
            df[col] = np.log1p(df[col])

    # Drop columns that are no longer needed
    # (we already used them to create new features)
    cols_to_drop = [
        'year_remod_add',
        'first_flr_sf', 'second_flr_sf', 'total_bsmt_sf',
        'full_bath', 'half_bath', 'bsmt_full_bath', 'bsmt_half_bath'
    ]
    df = df.drop(columns=cols_to_drop, errors='ignore')

    return df


# Run feature engineering
df_engineered = feature_engineering(df)

# Save result to database
df_engineered.to_sql('engineered_housing_data', engine, if_exists='replace', index=False)

print(f"Feature engineering done. Rows: {len(df_engineered)}")

Feature engineering done. Rows: 2930
